# Fase 2 — Variáveis derivadas fisicamente

Objetivo: construir e caracterizar combinações meteorológicas derivadas dos 12 canais ERA5 sem alterar o dataset de treinamento. As grandezas de diferença vetorial de vento são **proxies de cisalhamento em m/s**, não taxas de cisalhamento normalizadas por altura.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

def resolve_output(rel):
    candidates = [Path(rel), Path("..") / rel, Path.cwd() / rel, Path.cwd().parent / rel]
    for p in candidates:
        if p.exists():
            return p.resolve()
    return Path(rel).resolve()

OUT = resolve_output("analysis_outputs/02_derived")
print("Usando resultados em:", OUT)


In [ ]:
def load_parquet(name):
    path = OUT / name
    return pd.read_parquet(path) if path.exists() else pd.DataFrame()

def load_json(name):
    path = OUT / name
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

summary = load_json("analysis_summary.json")
formulas = load_parquet("formula_catalog.parquet")
derived = load_parquet("derived_summary.parquet")
quantiles = load_parquet("derived_quantiles.parquet")
hist = load_parquet("derived_histograms.parquet")
corr_long = load_parquet("derived_correlations.parquet")
samples_file = OUT / "derived_samples.npz"
samples = np.load(samples_file, allow_pickle=False) if samples_file.exists() else None


## 1. Catálogo das variáveis derivadas

In [ ]:
formulas

## 2. Resumo estatístico

In [ ]:
cols=["name","unit","sampled_pixel_count","mean","std","min","max","skewness_reservoir","excess_kurtosis_reservoir"]
derived[cols] if not derived.empty else derived

## 3. Quantis

In [ ]:
if not quantiles.empty:
    display(quantiles.pivot(index="name", columns="quantile", values="value"))

## 4. Distribuições das variáveis derivadas

In [ ]:
if samples is not None:
    names=list(samples.files)
    ncols=2
    nrows=int(np.ceil(len(names)/ncols))
    fig, axes=plt.subplots(nrows,ncols,figsize=(13,3.2*nrows))
    axes=np.asarray(axes).reshape(-1)
    for ax,name in zip(axes,names):
        vals=samples[name]
        ax.hist(vals,bins=80)
        ax.set_title(name)
        ax.set_ylabel("Contagem")
    for ax in axes[len(names):]: ax.axis("off")
    fig.tight_layout()
    plt.show()

## 5. Magnitude do vento nos três níveis

In [ ]:
if samples is not None:
    fig, ax=plt.subplots(figsize=(10,5))
    for name in ["wind_speed_10","wind_speed_850","wind_speed_500"]:
        vals=samples[name]
        ax.hist(vals,bins=100,density=True,histtype="step",label=name)
    ax.set_xlabel("Magnitude do vento (m/s)")
    ax.set_ylabel("Densidade")
    ax.legend()
    plt.show()

## 6. Estrutura vertical termodinâmica

In [ ]:
if samples is not None:
    fig, axes=plt.subplots(1,3,figsize=(15,4))
    for ax,name in zip(axes,["delta_t_500_850","delta_t_850_surface","delta_r_500_850"]):
        ax.hist(samples[name],bins=100)
        ax.set_title(name)
    fig.tight_layout(); plt.show()

## 7. Proxies de cisalhamento (diferença vetorial de vento)

In [ ]:
if samples is not None:
    fig, ax=plt.subplots(figsize=(10,5))
    for name in ["bulk_wind_diff_10_850","bulk_wind_diff_850_500"]:
        ax.hist(samples[name],bins=100,density=True,histtype="step",label=name)
    ax.set_xlabel("Diferença vetorial de vento (m/s)")
    ax.set_ylabel("Densidade")
    ax.legend(); plt.show()

## 8. Correlação entre diagnósticos derivados

Esta matriz serve apenas para identificar possível redundância entre diagnósticos; a análise multivariada formal será realizada mais adiante.

In [ ]:
if not corr_long.empty:
    mat=corr_long.pivot(index="var_a",columns="var_b",values="pearson_r")
    fig,ax=plt.subplots(figsize=(9,8))
    im=ax.imshow(mat.values,vmin=-1,vmax=1,aspect="auto")
    ax.set_xticks(range(len(mat.columns)),mat.columns,rotation=60,ha="right")
    ax.set_yticks(range(len(mat.index)),mat.index)
    fig.colorbar(im,ax=ax,label="Pearson r")
    fig.tight_layout(); plt.show()

## 9. Síntese da Fase 2

In [ ]:
check = pd.DataFrame([
    {"checagem":"Variáveis derivadas calculadas","status":"OK" if len(derived)>=8 else "REVISAR","detalhe":len(derived)},
    {"checagem":"Amostras para visualização","status":"OK" if samples is not None else "REVISAR","detalhe":str(samples_file)},
    {"checagem":"Catálogo de fórmulas","status":"OK" if len(formulas)>=8 else "REVISAR","detalhe":len(formulas)},
])
check